# Baseline sweep — exploratory analysis

Reads `baseline_sweep_main.csv`.

**Ground rules**

1. Never compare against the earlier pilot sweep. Three things differ (parametrization,
   clipping, schedule length), so any difference is uninterpretable.
2. Report distributions over seeds, not medians. One arm spans a 6x range.
3. Seeds are shared across arms, so comparisons are **paired**.
4. One comparison, one `MC_samples`. Mixing sample counts confounds the estimator
   comparison with a sample-size effect.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170, "display.max_columns", 40, "display.max_rows", 80)
raw = pd.read_csv("baseline_sweep_main.csv")

for c in ["seed", "n_steps"]:
    raw[c] = raw[c].astype(int)
for c in ["elbo", "kl", "elbo_mc_stderr", "ms_per_step", "nfe_per_step",
          "cumulative_train_s", "grad_norm_mean", "grad_norm_p95", "kwarg_MC_samples"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
for c in ["diverged", "unstable"]:
    raw[c] = raw[c].astype(str).str.lower().isin(["true", "1"])

FINAL = raw.n_steps.max()
print(f"{len(raw)} rows | {raw.config.nunique()} configs | seeds {sorted(raw.seed.unique())}")
print(f"checkpoints {sorted(raw.n_steps.unique())} | final = {FINAL}")

## 0. Filtering — do this once, use it everywhere

Three exclusions, each for a stated reason:

- **soft Gumbel-Softmax** — diverges on 4-5 of 5 seeds at every temperature, with gradient
  norms of order 1e17. Kept only in §2, where the divergence is the finding.
- **exact marginalization** — the reference gradient, not a competitor. Shown as a
  horizontal line, never as a bar in the ranking.
- **MC ablation arms** — different `MC_samples` from the main comparison. Analysed on their
  own in §8, where sample count is the independent variable.

`MC_MAIN = 256` is the sample count shared by every main arm. The reference is taken at the
same 256 (`exact_marg (MC=256)`), *not* the 1024 version, so the comparison is matched.

In [ ]:
MC_MAIN = 256

is_soft_gs = raw.config.str.contains("soft", case=False, na=False)
is_ref     = raw.ablation_group.astype(str).eq("exact_marg") | \
             raw.config.str.contains("Marginalization", case=False, na=False)
is_ablation_member = raw.ablation_group.notna() & (raw.kwarg_MC_samples != MC_MAIN)

MAIN = raw[~is_soft_gs & ~is_ref & ~is_ablation_member &
           (raw.kwarg_MC_samples == MC_MAIN)].copy()

# MC-matched reference, used only as a line.
REF = raw[is_ref & (raw.kwarg_MC_samples == MC_MAIN)].copy()
REF_FINAL = REF.loc[REF.n_steps == FINAL, "kl"]

# MC ablation: every arm belonging to an ablation family, soft GS excluded.
ABL = raw[raw.ablation_group.notna() & ~is_soft_gs].copy()

print(f"MAIN : {MAIN.config.nunique():>3} arms, all MC={MC_MAIN}")
print(f"REF  : {REF.config.unique()}  median KL at {FINAL} = {REF_FINAL.median():.4f}")
print(f"ABL  : {ABL.ablation_group.nunique()} families, "
      f"MC in {sorted(ABL.kwarg_MC_samples.dropna().unique().astype(int))}")
print(f"\nexcluded: {sorted(set(raw.config) - set(MAIN.config) - set(REF.config))}")

## 1. Completeness

Missing runs silently bias every summary. Check before anything else.

In [ ]:
chk = (MAIN.groupby("config")
          .agg(seeds=("seed", "nunique"), checkpoints=("n_steps", "nunique"),
               kl_missing=("kl", lambda s: s.isna().sum()),
               any_diverged=("diverged", "any")))
bad = chk[(chk.seeds != MAIN.seed.nunique()) |
          (chk.checkpoints != MAIN.n_steps.nunique()) |
          (chk.kl_missing > 0) | chk.any_diverged]
print("incomplete or diverged arms in MAIN:" if len(bad) else
      "MAIN is complete: every arm has all seeds and checkpoints, none diverged.")
display(bad)

## 2. Divergence — the only section that uses the excluded arms

Soft Gumbel-Softmax blows up at every temperature. The soft sample is a convex combination
lying *between* components, but it is scored against the density of the **hard** mixture,
which is near zero there — so the optimizer is rewarded for separating the components
further and the relaxed objective is unbounded above.

In [ ]:
div = (raw[raw.diverged].groupby("config")
          .agg(seeds_diverged=("seed", "nunique"))
          .join(raw.groupby("config").grad_norm_mean.median().rename("gradnorm_med"))
          .sort_values("seeds_diverged", ascending=False))
display(div)
print(f"\nMAIN arms diverged: {MAIN.diverged.sum()}  (expected 0)")

## 3. Spread over seeds — the robustness result

The strongest claim available from a single self-consistent experiment. Medians hide it.

In [ ]:
fin = MAIN[(MAIN.n_steps == FINAL) & MAIN.kl.notna()]

sp = (fin.groupby("config").kl
        .agg(n="size", min="min", median="median", max="max")
        .assign(ratio=lambda d: d["max"] / d["min"])
        .sort_values("median"))
display(sp.round(4))

In [ ]:
order = sp.index.tolist()
fig, ax = plt.subplots(figsize=(9, 0.32 * len(order) + 2))
for i, cfg in enumerate(order):
    v = fin.loc[fin.config == cfg, "kl"].values
    ax.plot([v.min(), v.max()], [i, i], lw=1, alpha=.35, color="grey", zorder=1)
    ax.scatter(v, np.full_like(v, i, dtype=float), s=30, alpha=.8, zorder=2)
ax.axvline(REF_FINAL.median(), color="crimson", ls="--", lw=1.2,
           label=f"exact marginalization (MC={MC_MAIN}) = {REF_FINAL.median():.4f}")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8)
ax.set_xscale("log"); ax.set_xlabel(f"KL(q||p) at {FINAL} steps  (one dot per seed)")
ax.legend(fontsize=8); ax.grid(axis="x", alpha=.3); plt.tight_layout(); plt.show()

## 4. Is anything converged?

Fit `log KL ~ log steps` over the last three checkpoints. Slope near -1 means KL still
halves per doubling of steps — the "final" number is a reading from a descending curve, and
**arms with different slopes will reorder given more budget**.

In [ ]:
def tail_slope(g, last=3):
    g = g.sort_values("n_steps").dropna(subset=["kl"]).tail(last)
    if len(g) < 2 or (g.kl <= 0).any():
        return np.nan
    return np.polyfit(np.log(g.n_steps), np.log(g.kl), 1)[0]

slopes = (MAIN.groupby(["config", "seed"]).apply(tail_slope, include_groups=False)
             .groupby("config").median().sort_values().rename("slope"))
ref_slope = (REF.groupby(["config", "seed"]).apply(tail_slope, include_groups=False)
                .median())
display(slopes.to_frame().round(2))
print(f"reference (exact marginalization) slope: {ref_slope:.2f}")
print("\n-1 => not converged at all;  0 => converged.")
print("Differing slopes => the ranking at this budget is not the ranking at a larger one.")

In [ ]:
sel = ["Score Function", "Score Function + RLOO", "ODE linear midpoint (steps=8)",
       "ODE geometric rk4 (steps=8)", "Straight-Through (tau=0.5)"]
fig, ax = plt.subplots(figsize=(8, 5))
for cfg in sel:
    g = MAIN[MAIN.config == cfg].groupby("n_steps").kl
    med = g.median()
    ax.plot(med.index, med.values, marker="o", label=cfg)
    ax.fill_between(med.index, g.min().values, g.max().values, alpha=.12)
r = REF.groupby("n_steps").kl.median()
ax.plot(r.index, r.values, color="crimson", ls="--", lw=1.4, label="exact marg. (reference)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("training step"); ax.set_ylabel("KL(q||p)")
ax.set_title(f"Learning curves, MC={MC_MAIN} throughout (median, min-max band)")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 5. Paired comparisons

Seeds are shared, so pair on seed. With five seeds this is the difference between no signal
and clear signal.

In [ ]:
def paired(a, b, steps=None):
    steps = steps or FINAL
    piv = (MAIN[MAIN.config.isin([a, b]) & (MAIN.n_steps == steps)]
             .pivot_table(index="seed", columns="config", values="kl").dropna())
    if a not in piv or b not in piv:
        return None
    return piv.assign(diff=piv[a] - piv[b], ratio=piv[a] / piv[b])

def report(a, b):
    p = paired(a, b)
    if p is None:
        print(f"{a} vs {b}: incomparable"); return
    print(f"{a}\n  vs {b}")
    print(f"  first lower on {(p['diff'] < 0).sum()}/{len(p)} seeds | "
          f"median ratio {p['ratio'].median():.2f} | median diff {p['diff'].median():+.4f}")
    display(p.round(4))

report("ODE linear midpoint (steps=8)", "Score Function")
report("ODE linear midpoint (steps=8)", "Straight-Through (tau=0.5)")
report("Score Function + RLOO", "Score Function")

In [ ]:
def boot_median_ci(v, B=20000, alpha=.05, seed=0):
    rng = np.random.default_rng(seed); v = np.asarray(v)
    return np.quantile(np.median(rng.choice(v, (B, len(v)), replace=True), axis=1),
                       [alpha / 2, 1 - alpha / 2])

ci = pd.DataFrame(
    [(c, g.kl.median(), *boot_median_ci(g.kl.values)) for c, g in fin.groupby("config")],
    columns=["config", "median", "lo", "hi"]).set_index("config")
ci["width"] = ci.hi - ci.lo
display(ci.sort_values("median").round(4))
print("Heavily overlapping intervals => do not claim one arm beats another.")

## 6. Solver and path ablation

All at MC=256, so the only thing varying is the integrator.

In [ ]:
ode = fin[fin.config.str.startswith("ODE")].copy()
ode["path"] = ode.kwarg_path
ode["solver"] = ode.kwarg_ode_solver
ode["steps"] = pd.to_numeric(ode.kwarg_ode_steps, errors="coerce")

display(ode.pivot_table(index=["solver", "steps"], columns="path",
                        values="kl", aggfunc="median").round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for (path, solver), g in ode.dropna(subset=["steps"]).groupby(["path", "solver"]):
    m = g.groupby("steps").kl.median().sort_index()
    ax.plot(m.index, m.values, marker="o", ls="-" if path == "linear" else "--",
            label=f"{path} {solver}")
ax.axhline(REF_FINAL.median(), color="crimson", ls=":", lw=1, label="reference")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("integration steps"); ax.set_ylabel("KL(q||p)")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 7. Cost: NFE, wall-clock, and rho

`rho = C_v / C_p` is the slope-to-intercept ratio of `ms_per_step` against NFE.

**rho is target-specific.** A value fitted on the banana says nothing about
Lotka-Volterra, where `C_p` is roughly 35x larger. Do not quote this number for LV.

In [ ]:
stage = {"euler": 1, "midpoint": 2, "rk4": 4}
f = ode.dropna(subset=["steps"]).copy()
f["nfe"] = f.steps * f.solver.map(stage)
fit = f.groupby(["path", "solver", "nfe"]).ms_per_step.median().reset_index()

print(f"{'path':11}{'solver':10}{'b (ms/NFE)':>12}{'a (ms)':>10}{'rho=b/a':>10}")
for (path, solver), g in fit.groupby(["path", "solver"]):
    if len(g) < 2: continue
    b, a = np.polyfit(g.nfe, g.ms_per_step, 1)
    print(f"{path:11}{solver:10}{b:12.4f}{a:10.4f}{b / a:10.2f}")
print("\nrho >> 1: the velocity field costs more than the target -- expected on the banana,")
print("and the wrong regime for the thesis's claim. LV is where rho < 1.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, xcol, xlabel in [(axes[0], "nfe_per_step", "NFE per gradient step"),
                         (axes[1], "ms_per_step", "ms per gradient step")]:
    for cfg, g in fin.groupby("config"):
        if g[xcol].isna().all(): continue
        x, y = g[xcol].median(), g.kl.median()
        ax.scatter(x, y, s=30)
        ax.annotate(cfg[:26], (x, y), fontsize=6, alpha=.75,
                    xytext=(3, 3), textcoords="offset points")
    ax.axhline(REF_FINAL.median(), color="crimson", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel("KL(q||p)"); ax.grid(alpha=.3)
axes[0].set_title("Accuracy vs transport cost (implementation-independent)")
axes[1].set_title("Accuracy vs wall-clock (implementation-dependent)")
plt.tight_layout(); plt.show()

## 8. Sample-count ablation

Here `MC_samples` **is** the independent variable, so the families are compared within
themselves and never against the MAIN arms. If the ODE gradient has lower variance per
sample it should win most at small `MC_samples`.

In [ ]:
abl = ABL[(ABL.n_steps == FINAL) & ABL.kl.notna()]
piv = abl.pivot_table(index="ablation_group", columns="kwarg_MC_samples",
                      values="kl", aggfunc="median")
display(piv.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
for fam, g in abl.groupby("ablation_group"):
    m = g.groupby("kwarg_MC_samples").kl.median().sort_index()
    ax.plot(m.index, m.values, marker="o", label=fam)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("MC_samples"); ax.set_ylabel("KL(q||p)")
ax.set_title("Same estimator, varying sample count")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 9. Would gradient clipping have bound?

Clipping was removed because at 1.0 it bound unequally across estimators, acting as a hidden
per-estimator learning-rate penalty. The replacement should be high enough never to activate
in normal operation.

In [ ]:
gn = (MAIN.groupby("config")[["grad_norm_mean", "grad_norm_p95"]]
         .median().sort_values("grad_norm_p95", ascending=False))
display(gn.round(3))

p95max = gn.grad_norm_p95.max()
print(f"\nlargest p95 over MAIN arms : {p95max:.2f}")
print(f"threshold 1.0 would bind for {(gn.grad_norm_p95 > 1.0).sum()}/{len(gn)} arms")
print(f"suggested threshold        : {10 * p95max:.1f}")

## 10. Scratch